In [101]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
dir_path = '/Users/RachidAJ/Desktop/Statistics Projects'

In [102]:
orders = pd.read_csv(f'{dir_path}/instacart-market-basket-analysis/orders.csv' )
order_products_train = pd.read_csv(f'{dir_path}/instacart-market-basket-analysis/order_products__train.csv')
order_products_prior = pd.read_csv(f'{dir_path}/instacart-market-basket-analysis/order_products__prior.csv')
products = pd.read_csv(f'{dir_path}/instacart-market-basket-analysis/products.csv')
aisles = pd.read_csv(f'{dir_path}/instacart-market-basket-analysis/aisles.csv')
departments = pd.read_csv(f'{dir_path}/instacart-market-basket-analysis/departments.csv')


In [103]:
orders[orders['eval_set'] != 'prior'].head()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
10,1187899,1,train,11,4,8,14.0
25,1492625,2,train,15,1,11,30.0
38,2774568,3,test,13,5,15,11.0
44,329954,4,test,6,3,12,30.0
49,2196797,5,train,5,0,11,6.0


In [104]:
def reduce_mem_usage(train_data):
    
#  iterate through all the columns of a dataframe and modify the data type to reduce memory usage."""
    start_mem = train_data.memory_usage().sum() / 1024**2
    print('Memory usage of dataframe is {:.2f} MB'.format(start_mem))

    for col in train_data.columns:
        col_type = train_data[col].dtype

        if col_type != object:
            c_min = train_data[col].min()
            c_max = train_data[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    train_data[col] = train_data[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    train_data[col] = train_data[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    train_data[col] = train_data[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    train_data[col] = train_data[col].astype(np.int64)  
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    train_data[col] = train_data[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    train_data[col] = train_data[col].astype(np.float32)
                else:
                    train_data[col] = train_data[col].astype(np.float64)
        else:
            train_data[col] = train_data[col].astype('category')
        end_mem = train_data.memory_usage().sum() / 1024**2
        print('Memory usage after optimization is: {:.2f} MB'.format(end_mem))
        print('Decreased by {:.1f}%'.format(100 * (start_mem - end_mem) / start_mem))

    return train_data

In [105]:
reduce_mem_usage(order_products_prior)
reduce_mem_usage(order_products_train)
reduce_mem_usage(products)
reduce_mem_usage(orders)
reduce_mem_usage(departments)
reduce_mem_usage(aisles)

Memory usage of dataframe is 989.82 MB
Memory usage after optimization is: 866.09 MB
Decreased by 12.5%
Memory usage after optimization is: 742.37 MB
Decreased by 25.0%
Memory usage after optimization is: 556.78 MB
Decreased by 43.7%
Memory usage after optimization is: 340.25 MB
Decreased by 65.6%
Memory usage of dataframe is 42.26 MB
Memory usage after optimization is: 36.97 MB
Decreased by 12.5%
Memory usage after optimization is: 31.69 MB
Decreased by 25.0%
Memory usage after optimization is: 22.45 MB
Decreased by 46.9%
Memory usage after optimization is: 13.20 MB
Decreased by 68.7%
Memory usage of dataframe is 1.52 MB
Memory usage after optimization is: 1.33 MB
Decreased by 12.5%
Memory usage after optimization is: 2.52 MB
Decreased by -66.5%
Memory usage after optimization is: 2.24 MB
Decreased by -47.7%
Memory usage after optimization is: 1.91 MB
Decreased by -25.8%
Memory usage of dataframe is 182.71 MB
Memory usage after optimization is: 169.66 MB
Decreased by 7.1%
Memory usage

,aisle_id,aisle
0,1,prepared soups salads
1,2,specialty cheeses
2,3,energy granola bars
3,4,instant foods
4,5,marinades meat preparation
...,...,...
129,130,hot cereal pancake mixes
130,131,dry pasta
131,132,beauty
132,133,muscles joints pain relief


In [106]:
import datetime
import math

def datetime_to_radians(x):
    # radians are calculated using a 24-hour circle, not 12-hour, starting at north and moving clockwise
    seconds_from_midnight = 3600 * x
    radians = float(seconds_from_midnight) / float(12 * 60 * 60) * 2.0 * math.pi
    return radians

def average_angle(angles):
    # angles measured in radians
    x_sum = np.sum(np.sin(angles))
    y_sum = np.sum(np.cos(angles))
    x_mean = x_sum / float(len(angles))
    y_mean = y_sum / float(len(angles))
    return np.arctan2(x_mean, y_mean)

def radians_to_time_of_day(x):
    # radians are measured clockwise from north and represent time in a 24-hour circle
    seconds_from_midnight = int(float(x) / (2.0 * math.pi) * 12.0 * 60.0 * 60.0)
    hour = seconds_from_midnight // 3600 % 24
    minute = (seconds_from_midnight % 3600) // 60
    second = seconds_from_midnight % 60
    return datetime.time(hour, minute, second)
    
def average_times_of_day(x):
    # input datetime.datetime array and output datetime.time value
    angles = [datetime_to_radians(y) for y in x]
    avg_angle = average_angle(angles)
    return radians_to_time_of_day(avg_angle)

def day_to_radians(day):
    radians = float(day) / float(7) * 2.0 * math.pi
    return radians
def radians_to_days(x):
    day = int(float(x) / (2.0 * math.pi) * 7) % 7
    return day
def average_days(x):
    angles = [day_to_radians(y) for y in x]
    avg_angle = average_angle(angles)
    return radians_to_days(avg_angle)

# Predictor Features

In [107]:
users = orders[orders['eval_set'] == 'prior']
users['days_since_prior_order'].dropna()
users = users.groupby('user_id').agg(
 user_orders= ('order_number' , max),
 user_period=('days_since_prior_order', sum),
 user_mean_days_since_prior = ('days_since_prior_order','mean') 
)
users.head()

/var/folders/_j/yqtls3w13gbg4tfd7g2xq1h40000gp/T/ipykernel_6376/1220139000.py:3: FutureWarning: The provided callable <built-in function max> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  users = users.groupby('user_id').agg(
/var/folders/_j/yqtls3w13gbg4tfd7g2xq1h40000gp/T/ipykernel_6376/1220139000.py:3: FutureWarning: The provided callable <built-in function sum> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  users = users.groupby('user_id').agg(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,user_orders,user_period,user_mean_days_since_prior
user_id,,,
1,10,176.0,19.555555
2,14,198.0,15.230769
3,12,133.0,12.090909
4,5,55.0,13.750000
5,4,40.0,13.333333


In [108]:

orders_products =pd.merge(orders , order_products_prior, on='order_id', how='inner')
grouped_orders_products = orders_products.groupby(['order_id']).agg(
    basket_size = ('product_id', 'count')
).reset_index()
orders_products = orders_products.merge(grouped_orders_products, on='order_id', how='left')
orders_products.head()

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,add_to_cart_order,reordered,basket_size
0,2539329,1,prior,1,2,8,NaN,196,1,0,5
1,2539329,1,prior,1,2,8,NaN,14084,2,0,5
2,2539329,1,prior,1,2,8,NaN,12427,3,0,5
3,2539329,1,prior,1,2,8,NaN,26088,4,0,5
4,2539329,1,prior,1,2,8,NaN,26405,5,0,5


In [109]:
orders_products['p_reordered']= orders_products['reordered']==1
orders_products['non_first_order']= orders_products['order_number']>1

us=orders_products
us=orders_products.groupby('user_id').agg(
    
     user_total_products =('user_id','count') ,
     p_reordered =('p_reordered', sum) ,
     non_first_order =('non_first_order', sum),
     user_distinct_products=('product_id','nunique')

).reset_index()
us['user_reorder_ratio']=us['p_reordered']/us['non_first_order']

del us["p_reordered"],us["non_first_order"]
del orders_products['p_reordered' ],orders_products['non_first_order']

us.head()

/var/folders/_j/yqtls3w13gbg4tfd7g2xq1h40000gp/T/ipykernel_6376/2019911228.py:5: FutureWarning: The provided callable <built-in function sum> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  us=orders_products.groupby('user_id').agg(


,user_id,user_total_products,user_distinct_products,user_reorder_ratio
0,1,59,18,0.759259
1,2,195,102,0.510989
2,3,88,33,0.705128
3,4,18,17,0.071429
4,5,37,23,0.538462


In [110]:
users =pd.merge(users,us ,on='user_id',  how='inner')
users['user_average_basket'] = users['user_total_products'] / users['user_orders']
users.head()

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,user_id,user_orders,user_period,user_mean_days_since_prior,user_total_products,user_distinct_products,user_reorder_ratio,user_average_basket
0,1,10,176.0,19.555555,59,18,0.759259,5.900000
1,2,14,198.0,15.230769,195,102,0.510989,13.928571
2,3,12,133.0,12.090909,88,33,0.705128,7.333333
3,4,5,55.0,13.750000,18,17,0.071429,3.600000
4,5,4,40.0,13.333333,37,23,0.538462,9.250000


In [111]:
us = orders[orders['eval_set'] != 'prior']
us['time_since_last_order'] = us['days_since_prior_order']
us['future_order_dow'] = us['order_dow']
us['future_order_hour_of_day'] = us['order_hour_of_day']

us = us[['user_id','order_id','eval_set','time_since_last_order', 'future_order_dow', 'future_order_hour_of_day']]
users_features = pd.merge(users , us, on='user_id', how='inner') 
del us, users

users_features.head()

/var/folders/_j/yqtls3w13gbg4tfd7g2xq1h40000gp/T/ipykernel_6376/4059787326.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  us['time_since_last_order'] = us['days_since_prior_order']
/var/folders/_j/yqtls3w13gbg4tfd7g2xq1h40000gp/T/ipykernel_6376/4059787326.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  us['future_order_dow'] = us['order_dow']
/var/folders/_j/yqtls3w13gbg4tfd7g2xq1h40000gp/T/ipykernel_6376/4059787326.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice fro

,user_id,user_orders,user_period,user_mean_days_since_prior,user_total_products,user_distinct_products,user_reorder_ratio,user_average_basket,order_id,eval_set,time_since_last_order,future_order_dow,future_order_hour_of_day
0,1,10,176.0,19.555555,59,18,0.759259,5.900000,1187899,train,14.0,4,8
1,2,14,198.0,15.230769,195,102,0.510989,13.928571,1492625,train,30.0,1,11
2,3,12,133.0,12.090909,88,33,0.705128,7.333333,2774568,test,11.0,5,15
3,4,5,55.0,13.750000,18,17,0.071429,3.600000,329954,test,30.0,3,12
4,5,4,40.0,13.333333,37,23,0.538462,9.250000,2196797,train,6.0,0,11


In [112]:
prod_features = orders_products.groupby(['product_id']).agg(
    prod_freq = ('order_id', 'count'),
    prod_avg_position = ('add_to_cart_order', 'mean')
).reset_index()

prod_features.head()

,product_id,prod_freq,prod_avg_position
0,1,1852,5.801836
1,2,90,9.888889
2,3,277,6.415162
3,4,329,9.507599
4,5,15,6.466667


In [113]:
orders_products

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,add_to_cart_order,reordered,basket_size
0,2539329,1,prior,1,2,8,NaN,196,1,0,5
1,2539329,1,prior,1,2,8,NaN,14084,2,0,5
2,2539329,1,prior,1,2,8,NaN,12427,3,0,5
3,2539329,1,prior,1,2,8,NaN,26088,4,0,5
4,2539329,1,prior,1,2,8,NaN,26405,5,0,5
...,...,...,...,...,...,...,...,...,...,...,...
32434484,2977660,206209,prior,13,1,12,7.0,14197,5,1,9
32434485,2977660,206209,prior,13,1,12,7.0,38730,6,0,9
32434486,2977660,206209,prior,13,1,12,7.0,31477,7,0,9
32434487,2977660,206209,prior,13,1,12,7.0,6567,8,0,9


In [114]:
non_first_order = orders_products['order_number'] != 1

groupedorders_products = orders_products[non_first_order].groupby(['product_id']).agg(
    prod_reorder_ratio = ('reordered', 'mean')
).reset_index()

prod_features = prod_features.merge(groupedorders_products, on='product_id', how='left')
groupedorders_products = orders_products[non_first_order].groupby(['product_id', 'user_id']).agg(
    user_prod_freq = ('order_id', 'count')
).reset_index()
groupedorders_products = groupedorders_products.groupby(['product_id']).agg(
    user_prod_avg_freq = ('user_prod_freq', 'mean')
).reset_index()
prod_features = prod_features.merge(groupedorders_products, on='product_id', how='left')

del groupedorders_products, non_first_order
prod_features.head()

,product_id,prod_freq,prod_avg_position,prod_reorder_ratio,user_prod_avg_freq
0,1,1852,5.801836,0.647662,2.641566
1,2,90,9.888889,0.137931,1.160000
2,3,277,6.415162,0.780769,3.714286
3,4,329,9.507599,0.506897,1.746988
4,5,15,6.466667,0.642857,2.333333


In [115]:
data = orders_products

data = data.groupby(['user_id','product_id']).agg(
 up_orders= ('product_id', 'count'),
 up_first_order=('order_number', min),
 up_last_order = ('order_number',max),
 up_average_cart_position = ('add_to_cart_order','mean')
).reset_index()
 
del orders_products
data.head()

/var/folders/_j/yqtls3w13gbg4tfd7g2xq1h40000gp/T/ipykernel_6376/1986339743.py:3: FutureWarning: The provided callable <built-in function min> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  data = data.groupby(['user_id','product_id']).agg(
/var/folders/_j/yqtls3w13gbg4tfd7g2xq1h40000gp/T/ipykernel_6376/1986339743.py:3: FutureWarning: The provided callable <built-in function max> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  data = data.groupby(['user_id','product_id']).agg(


,user_id,product_id,up_orders,up_first_order,up_last_order,up_average_cart_position
0,1,196,10,1,10,1.400000
1,1,10258,9,2,10,3.333333
2,1,10326,1,5,5,5.000000
3,1,12427,10,1,10,3.300000
4,1,13032,3,2,10,6.333333


In [116]:
data = data.merge(users_features[['user_id','user_orders']], on='user_id' , how='left')
data['up_order_rate'] = data['up_orders']/data['user_orders']
data['up_orders_since_last_order'] = data['user_orders'] - data['up_last_order']
data['up_order_rate_since_first_order'] = data['up_orders']/(data['user_orders'] - data['up_first_order'] + 1)
del data['user_orders']

data.head()

,user_id,product_id,up_orders,up_first_order,up_last_order,up_average_cart_position,up_order_rate,up_orders_since_last_order,up_order_rate_since_first_order
0,1,196,10,1,10,1.400000,1.0,0,1.000000
1,1,10258,9,2,10,3.333333,0.9,0,1.000000
2,1,10326,1,5,5,5.000000,0.1,5,0.166667
3,1,12427,10,1,10,3.300000,1.0,0,1.000000
4,1,13032,3,2,10,6.333333,0.3,0,0.333333


In [117]:
data = data.merge(users_features, on='user_id', how='left').merge(prod_features, on='product_id', how='left')
del users_features, prod_features
order_products_future = order_products_train.merge(orders, on='order_id', how='left')
order_products_future = order_products_future[['user_id', 'product_id', 'reordered']]
data = data.merge(order_products_future, on=['user_id', 'product_id'], how='left')
data['reordered'].fillna(0, inplace = True)

/var/folders/_j/yqtls3w13gbg4tfd7g2xq1h40000gp/T/ipykernel_6376/2019572090.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['reordered'].fillna(0, inplace = True)


In [118]:
data.head()
data.isna().sum()
data.fillna(data.mean(numeric_only=True), inplace=True)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pandas/core/nanops.py:1496: RuntimeWarning: overflow encountered in cast
  return count.astype(dtype, copy=False)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/numpy/_core/_methods.py:52: RuntimeWarning: overflow encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)


In [119]:
data.isna().sum()

user_id                            0
product_id                         0
up_orders                          0
up_first_order                     0
up_last_order                      0
up_average_cart_position           0
up_order_rate                      0
up_orders_since_last_order         0
up_order_rate_since_first_order    0
user_orders                        0
user_period                        0
user_mean_days_since_prior         0
user_total_products                0
user_distinct_products             0
user_reorder_ratio                 0
user_average_basket                0
order_id                           0
eval_set                           0
time_since_last_order              0
future_order_dow                   0
future_order_hour_of_day           0
prod_freq                          0
prod_avg_position                  0
prod_reorder_ratio                 0
user_prod_avg_freq                 0
reordered                          0
dtype: int64

In [120]:
data.columns

Index(['user_id', 'product_id', 'up_orders', 'up_first_order', 'up_last_order',
       'up_average_cart_position', 'up_order_rate',
       'up_orders_since_last_order', 'up_order_rate_since_first_order',
       'user_orders', 'user_period', 'user_mean_days_since_prior',
       'user_total_products', 'user_distinct_products', 'user_reorder_ratio',
       'user_average_basket', 'order_id', 'eval_set', 'time_since_last_order',
       'future_order_dow', 'future_order_hour_of_day', 'prod_freq',
       'prod_avg_position', 'prod_reorder_ratio', 'user_prod_avg_freq',
       'reordered'],
      dtype='object')

In [122]:
X_train = data[data['eval_set'] == 'train']
y_train = X_train['reordered']
X_test = data[data['eval_set'] == 'test']

In [123]:
from sklearn.model_selection import train_test_split

print('Class distribution before splitting')
pos_count = np.sum(X_train['reordered']==1)
neg_count = np.sum(X_train['reordered']==0)
print('positive ratio: ', pos_count)
print('negative count: ', neg_count)
print('positive ratio: ', pos_count/(pos_count+neg_count))

X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.3)
print('Class distribution of Train set')
train_pos_count = np.sum(X_train['reordered']==1)
train_neg_count = np.sum(X_train['reordered']==0)
print('positive count: ', train_pos_count)
print('negative count: ', train_neg_count)
print('positive ratio: ', train_pos_count/(train_pos_count+train_neg_count))

print('Class distribution of Validation set')
val_pos_count = np.sum(X_val['reordered']==1)
val_neg_count = np.sum(X_val['reordered']==0)
print('positive count: ', val_pos_count)
print('negative count: ', val_neg_count)
print('positive ratio: ', val_pos_count/(val_pos_count+val_neg_count))

# Removing eval_set and the target variable from features
X_train_non_pred_vars = X_train[['product_id', 'order_id', 'user_id']]
X_train.drop(['reordered', 'eval_set', 'product_id', 'order_id', 'user_id'], axis=1, inplace=True)

X_val_non_pred_vars = X_val[['product_id', 'order_id', 'user_id']]
X_val.drop(['reordered', 'eval_set', 'product_id', 'order_id', 'user_id'], axis=1, inplace=True)

X_test_non_pred_vars = X_test[['product_id', 'order_id', 'user_id']]
X_test.drop(['reordered', 'eval_set', 'product_id', 'order_id', 'user_id'], axis=1, inplace=True)

# Drop features I suspect redundant or of no significant importance as the feature importance graph says
X_train.drop(['user_total_products', 'user_distinct_products'], axis=1, inplace=True)
X_test.drop([ 'user_total_products', 'user_distinct_products'], axis=1, inplace=True)
X_val.drop(['user_total_products', 'user_distinct_products'], axis=1, inplace=True)

Class distribution before splitting
positive ratio:  828824
negative count:  7645837
positive ratio:  0.09780025419305857
Class distribution of Train set
positive count:  580206
negative count:  5352056
positive ratio:  0.09780518797045713
Class distribution of Validation set
positive count:  248618
negative count:  2293781
positive ratio:  0.09778874205032334


/var/folders/_j/yqtls3w13gbg4tfd7g2xq1h40000gp/T/ipykernel_6376/11717901.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test.drop(['reordered', 'eval_set', 'product_id', 'order_id', 'user_id'], axis=1, inplace=True)
/var/folders/_j/yqtls3w13gbg4tfd7g2xq1h40000gp/T/ipykernel_6376/11717901.py:37: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test.drop([ 'user_total_products', 'user_distinct_products'], axis=1, inplace=True)


In [124]:
print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape)

(5932262, 14) (5932262,)
(2542399, 14) (2542399,)
(4833292, 14)


# Naive Bayes

In [125]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, log_loss)
nb = GaussianNB()
nb.fit(X_train, y_train)

# --- Predictions ---
y_pred = nb.predict(X_val)
y_prob = nb.predict_proba(X_val)[:, 1]

# --- Evaluation ---
print("Naive Bayes Performance:")
print("Accuracy:", accuracy_score(y_val, y_pred))
print("Precision:", precision_score(y_val, y_pred))
print("Recall:", recall_score(y_val, y_pred))
print("F1 Score:", f1_score(y_val, y_pred))
print("ROC AUC:", roc_auc_score(y_val, y_prob))
print("Log Loss:", log_loss(y_val, y_prob))

Naive Bayes Performance:
Accuracy: 0.8859502383378848
Precision: 0.3394086297176774
Recall: 0.17572339894939223
F1 Score: 0.2315602009879789
ROC AUC: 0.7254758336706563
Log Loss: 0.5895532517022744


# Logistic Regression

In [126]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(max_iter=1000, solver='liblinear')  # You can change solver as needed
lr.fit(X_train, y_train)

# --- Predictions ---
y_pred = lr.predict(X_val)
y_prob = lr.predict_proba(X_val)[:, 1]

# --- Evaluation ---
print("Logistic Regression Performance:")
print("Accuracy:", accuracy_score(y_val, y_pred))
print("Precision:", precision_score(y_val, y_pred))
print("Recall:", recall_score(y_val, y_pred))
print("F1 Score:", f1_score(y_val, y_pred))
print("ROC AUC:", roc_auc_score(y_val, y_prob))
print("Log Loss:", log_loss(y_val, y_prob))

Logistic Regression Performance:
Accuracy: 0.9034105189626018
Precision: 0.5434342858770905
Recall: 0.07672010876123209
F1 Score: 0.13445792814670957
ROC AUC: 0.7611822801326829
Log Loss: 0.2799746956236753


In [127]:
import statsmodels.api as sm
logit_model = sm.Logit(y_train, X_train)
result = logit_model.fit()
print(result.summary())

Optimization terminated successfully.
         Current function value: 0.268143
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:              reordered   No. Observations:              5932262
Model:                          Logit   Df Residuals:                  5932248
Method:                           MLE   Df Model:                           13
Date:                Sun, 11 May 2025   Pseudo R-squ.:                  0.1627
Time:                        21:38:49   Log-Likelihood:            -1.5907e+06
converged:                       True   LL-Null:                   -1.8997e+06
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                      coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
up_orders                           0.0502      0.000    103.032  

In [128]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, log_loss

# --- Prédictions ---
y_prob = result.predict(X_val)  # Probabilités prévues (entre 0 et 1)
y_pred = (y_prob >= 0.5).astype(int)  # Seuil à 0.5 pour prédiction binaire

# --- Évaluation ---
print("Logistic Regression (statsmodels) Performance:")
print("Accuracy:", accuracy_score(y_val, y_pred))
print("Precision:", precision_score(y_val, y_pred))
print("Recall:", recall_score(y_val, y_pred))
print("F1 Score:", f1_score(y_val, y_pred))
print("ROC AUC:", roc_auc_score(y_val, y_prob))
print("Log Loss:", log_loss(y_val, y_prob))

Logistic Regression (statsmodels) Performance:
Accuracy: 0.9059478862287155
Precision: 0.572355593468194
Recall: 0.1511314546814792
F1 Score: 0.23912227221525714
ROC AUC: 0.7839384638063798
Log Loss: 0.2679614215304603


In [52]:
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

# Suppose que X est ton DataFrame de variables explicatives (sans la cible)
X_vif = add_constant(X_train)  # Ajout de la constante si nécessaire

# Calcul du VIF
vif_data = pd.DataFrame()
vif_data['Variable'] = X_vif.columns
vif_data['VIF'] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]

print(vif_data)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


                           Variable         VIF
0                             const  139.642621
1                         up_orders    3.249297
2                    up_first_order    4.037762
3                     up_last_order         inf
4          up_average_cart_position    1.606657
5                     up_order_rate    3.128689
6        up_orders_since_last_order         inf
7   up_order_rate_since_first_order    2.014208
8                       user_orders         inf
9                       user_period    3.364737
10       user_mean_days_since_prior    2.522818
11               user_reorder_ratio    2.146413
12              user_average_basket    1.681758
13            time_since_last_order    1.398963
14                 future_order_dow    1.003146
15         future_order_hour_of_day    1.001445
16                        prod_freq    1.857969
17                prod_avg_position    1.738748
18               prod_reorder_ratio    5.239718
19               user_prod_avg_freq    8

In [121]:
data.drop(['user_prod_avg_freq', 'user_orders','up_last_order','up_orders_since_last_order','prod_reorder_ratio'], axis=1, inplace=True)